# LOKLOK Ground-Truth Pipeline — Researcher Self-Report

**Thesis:** *The Social Media Radio Innovation and Its Impact on University Students' Ratings*
**Researcher:** Sherif Mohamed Abd Elmonem Rashwan (Teaching Assistant, Radio & TV, ACU)
**Framework:** Epistemic Systems Theory (EST) · **Platform:** LOKLOK
**Evidence role:** this notebook turns a recorded **researcher self-report** (the researcher's own teaching-assistant account of how students actually behave, commute, trust, verify, and decide whether to speak) into a machine-readable **ground-truth source** that can be triangulated with the Uber-rider records, the survey corpus, and the focus-group discussion, and that is **ready for Gaussian-Mixture (GMM) clustering and Monte-Carlo analysis.**

## What this notebook does, end-to-end

1. **Transcribes** the audio (Whisper Large-V3, VAD, Arabic↔English code-switching, word timestamps).
2. **Profiles the voice** (pitch, energy, pauses, speech rate) — the *"how"* layer of reflexivity.
3. **Embeds every utterance** into a dense multilingual vector (768-d, matching the thesis domain model `h_s ∈ R^768`) plus a full-document embedding and an anchored "claim book".
4. **Runs reflexive thematic analysis (Braun & Clarke, 6 phases)** on the transcript using Qwen 2.5 (4-bit), producing an auditable codebook, named themes, timestamped quotations, and a reflexivity/positionality statement.
5. **Maps every theme to EST-P1…P4**, the LOKLOK design (BRS, MORE, mandatory co-hosting, University-ID gate, Echo Meter), and the Podcast Paradox.
6. **GMM readiness:** BIC-selected Gaussian-Mixture clustering of the segment embeddings.
7. **Monte-Carlo readiness:** bootstrap confidence intervals for theme/cluster prevalence plus an embedding-driven Echo-Meter/MORE trigger simulation with Monte-Carlo standard errors.
8. **Compiles the deliverables:** a comprehensive LaTeX report (paste-ready for the thesis appendix), a Markdown mirror, a JSON archive, and `.npy` embedding artifacts.

## Hardware this notebook is tuned for

- **GPU: NVIDIA T4 (≈15–16 GB VRAM)** — Whisper in `float16`, LLM in **4-bit NF4** with double quantization.
- **RAM: ≈12 GB** — models are **load → use → free** (never two big models resident at once), with garbage collection and CUDA cache emptying between stages.
- If the 7B model ever fails to load, the notebook **automatically falls back to Qwen2.5-3B** and reports it.

## Honesty guard (thesis-consistent)

The outputs are **researcher-reflexivity ground truth** produced from *one* self-report recording: not a survey sample, not a human treatment effect, not a completed experiment. They are interpretive evidence ready for triangulation, and GMM/Monte-Carlo **readiness** objects that feed the thesis's reserved G0/G1 replication tables — not causal claims. Every inference below is either marked as observation, interpretation, or design implication.

In [ ]:
# -------------------------------------------------------------------------
# CELL 1 - Install dependencies (T4-optimised, minimal footprint)
# -------------------------------------------------------------------------
!pip install -q --upgrade pip
!pip install -q faster-whisper ctranslate2 librosa soundfile
!pip install -q transformers accelerate bitsandbytes sentence-transformers scikit-learn matplotlib
!apt-get -qq -y install ffmpeg > /dev/null 2>&1
print("[OK] Dependencies installed.")

In [ ]:
# -------------------------------------------------------------------------
# CELL 2 - Hardware check (must show a T4-class GPU)
# -------------------------------------------------------------------------
import os
import platform

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch  # noqa: E402

print("python:", platform.python_version())
print("torch :", torch.__version__, "| cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Menu: Runtime -> Change runtime type -> T4 GPU, then re-run.")
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"gpu   : {gpu_name} ({vram_gb:.1f} GB VRAM)")
if vram_gb < 13:
    print("[WARN] VRAM looks low; use Runtime -> Factory reset runtime if crashes occur.")

!nvidia-smi

In [ ]:
# -------------------------------------------------------------------------
# CELL 3 - Mount Google Drive (audio + outputs live here)
# -------------------------------------------------------------------------
from google.colab import drive

drive.mount("/content/drive")
print("[OK] Drive mounted.")

In [ ]:
# -------------------------------------------------------------------------
# CELL 4 - Configuration (edit paths here if needed)
# -------------------------------------------------------------------------
import gc
import glob
import json
import random
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DRIVE_ROOT = "/content/drive/MyDrive"
AUDIO_FILE_NAME = "selfreport"                            # upload selfreport.mp3 (no extension)
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "thesis_ground_truth_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Locate the audio file (mp3 / m4a / wav) with a forgiving search ------
AUDIO_PATH = os.path.join(DRIVE_ROOT, AUDIO_FILE_NAME + ".mp3")
if not os.path.exists(AUDIO_PATH):
    patterns = [
        os.path.join(DRIVE_ROOT, AUDIO_FILE_NAME + ".*"),
        os.path.join(DRIVE_ROOT, "*selfreport*.*"),
        os.path.join(DRIVE_ROOT, "*self*report*.*"),
        os.path.join(DRIVE_ROOT, "*self*talk*.*"),
        os.path.join(DRIVE_ROOT, "*self*.mp3"),
        os.path.join(DRIVE_ROOT, "*self*.m4a"),
        os.path.join(DRIVE_ROOT, "*self*.wav"),
    ]
    hits = []
    for pat in patterns:
        hits += [p for p in glob.glob(pat) if os.path.isfile(p)]
    hits = sorted({os.path.abspath(p) for p in hits}, key=os.path.getsize, reverse=True)
    if hits:
        AUDIO_PATH = hits[0]
    else:
        raise FileNotFoundError(
            f"No audio found in {DRIVE_ROOT} for name '{AUDIO_FILE_NAME}'. Upload the "
            f"self-report recording there, or edit AUDIO_FILE_NAME in this cell."
        )

SIZE_MB = os.path.getsize(AUDIO_PATH) / 1e6
print(f"[OK] Audio: {AUDIO_PATH} ({SIZE_MB:.1f} MB)")
print(f"[OK] Output dir: {OUTPUT_DIR}")

# --- Optional sibling ground-truth sources (Uber corpus / FGD export / survey) ---
# Any that exist are read for triangulation in the final report; missing ones are skipped.
EXTRA_SOURCE_PATHS = [
    os.path.join(DRIVE_ROOT, "uber_corpus"),
    os.path.join(DRIVE_ROOT, "fgd_analysis_parts.json"),
    os.path.join(DRIVE_ROOT, "qualitative_records.csv"),
    os.path.join(DRIVE_ROOT, "survey.csv"),
    os.path.join(DRIVE_ROOT, "coding_decisions.csv"),
]
EXISTING_EXTRA = [p for p in EXTRA_SOURCE_PATHS if os.path.exists(p)]
print(f"[i] Triangulation sources found: {len(EXISTING_EXTRA)}")

# --- Model choices ---------------------------------------------------------
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"          # 4-bit -> ~4-5 GB VRAM
LLM_FALLBACK_ID = "Qwen/Qwen2.5-3B-Instruct"       # automatic fallback on OOM
EMBEDDING_MODEL_ID = "intfloat/multilingual-e5-base"            # 768-d, matches thesis R^768
EMBEDDING_FALLBACK_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # 384-d
WHISPER_ID = "large-v3"

MAX_CHAPTER_TOKENS = 1500          # per LLM generation cap (T4 speed / VRAM)
CHUNK_CHAR_BUDGET = 4500           # transcript chunk size fed to the LLM
SEED = 42

In [ ]:
# -------------------------------------------------------------------------
# CELL 5 - Whisper Large-V3 transcription (AR/EN code-switch, VAD, timestamps)
# -------------------------------------------------------------------------
from faster_whisper import WhisperModel


def load_whisper():
    try:
        return WhisperModel(WHISPER_ID, device="cuda", compute_type="float16")
    except Exception as exc:  # rare VRAM pressure -> int8 hybrid
        print("[i] Whisper fallback to int8_float16:", exc)
        return WhisperModel(WHISPER_ID, device="cuda", compute_type="int8_float16")


whisper_model = load_whisper()
print("[i] Whisper loaded.")

t_start = time.time()
segments, info = whisper_model.transcribe(
    AUDIO_PATH,
    language=None,
    beam_size=5,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
    word_timestamps=True,
    condition_on_previous_text=False,
)

transcript_data = []
for seg in segments:
    words = []
    for w in (seg.words or []):
        try:
            words.append({"w": w.word, "s": round(float(w.start), 2),
                          "e": round(float(w.end), 2), "p": round(float(w.probability), 3)})
        except Exception:
            pass
    transcript_data.append({
        "start": round(float(seg.start), 2),
        "end": round(float(seg.end), 2),
        "text": seg.text.strip(),
        "lang": seg.language,
        "lang_p": round(float(seg.language_probability), 3),
        "words": words,
    })

full_plain = " ".join(t["text"] for t in transcript_data).strip()
ts_lines = [f"[{t['start']:.1f}s-{t['end']:.1f}s|{t['lang']}] {t['text']}"
            for t in transcript_data]
TS_TEXT = "\n".join(ts_lines)

n_words = sum(len(t["words"]) for t in transcript_data if t["words"])
if n_words == 0:
    n_words = len(full_plain.split())
speech_sec = sum(t["end"] - t["start"] for t in transcript_data)
audio_dur = float(info.duration)
wpm = (n_words / speech_sec * 60.0) if speech_sec > 0 else 0.0

lang_counts = {}
for t in transcript_data:
    lang_counts[t["lang"]] = lang_counts.get(t["lang"], 0) + 1

transcript_stats = {
    "audio_duration_sec": round(audio_dur, 2),
    "speech_duration_sec": round(speech_sec, 2),
    "segments": len(transcript_data),
    "words": n_words,
    "words_per_minute": round(wpm, 1),
    "language_segments": lang_counts,
    "detected_language": info.language,
    "detected_language_score": round(float(info.language_probability), 3),
}
print(f"[OK] Transcription done in {time.time() - t_start:.0f}s.")
print(json.dumps(transcript_stats, indent=2, ensure_ascii=False))

with open(os.path.join(OUTPUT_DIR, "researcher_self_report_transcript.json"), "w", encoding="utf-8") as f:
    json.dump({"stats": transcript_stats, "segments": transcript_data}, f,
              ensure_ascii=False, indent=2)

# --- Free Whisper before loading anything else (12 GB RAM discipline) -----
del whisper_model
gc.collect()
torch.cuda.empty_cache()
print("[OK] Whisper freed from GPU.")

In [ ]:
# -------------------------------------------------------------------------
# CELL 6 - Acoustic & paralinguistic profiling (the "how" layer)
# -------------------------------------------------------------------------
import librosa

y, sr = librosa.load(AUDIO_PATH, sr=16000, mono=True)
dur = len(y) / sr

# 1) Pitch F0 (voice range) - expressiveness / emphasis
f0, voiced, _ = librosa.pyin(y, fmin=80.0, fmax=1100.0, sr=sr)
f0v = f0[~np.isnan(f0)]

# 2) Energy / loudness (percentile-based -> robust to silences)
frame_length = int(0.025 * sr)
hop = int(0.010 * sr)
rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop)[0]
rms_db_all = librosa.amplitude_to_db(rms, ref=np.max)
energy_low = np.percentile(rms_db_all, 10)
energy_high = np.percentile(rms_db_all, 90)
energy_range_db = float(energy_high - energy_low)

# 3) Tempo / speaking rhythm (representative 60 s window - fast on 1h files)
y_tempo = y[30 * sr : 90 * sr] if len(y) > 120 * sr else y
tempo_arr = librosa.feature.tempo(y=y_tempo, sr=sr)
tempo_bpm = float(np.mean(tempo_arr))

# 4) Meaningful pauses (reflection / cognitive load)
silence_th = np.percentile(rms, 15)
is_sil = rms < silence_th
pauses, in_pause, ps = [], False, None
for i, s in enumerate(is_sil):
    at = i * hop / sr
    if s and not in_pause:
        in_pause = True
        ps = at
    elif (not s) and in_pause:
        in_pause = False
        if at - ps >= 0.6:
            pauses.append({"start": round(ps, 2), "duration": round(at - ps, 2)})
        ps = None
if in_pause and ps is not None and (dur - ps) >= 0.6:
    pauses.append({"start": round(ps, 2), "duration": round(dur - ps, 2)})

# 5) Voicing ratio + arousal (high-energy speaking fraction)
speech_frames = int(is_sil.size) - int(is_sil.sum())
voiced_ratio = speech_frames / is_sil.size if is_sil.size else 0.0
arousal = float(np.mean(rms_db_all > np.percentile(rms_db_all, 80)))

acoustic_profile = {
    "duration_sec": round(dur, 2),
    "pitch_mean_hz": round(float(np.mean(f0v)), 1) if len(f0v) else 0.0,
    "pitch_std_hz": round(float(np.std(f0v)), 1) if len(f0v) else 0.0,
    "energy_range_db": energy_range_db,
    "tempo_bpm": tempo_bpm,
    "significant_pauses_count": len(pauses),
    "total_reflection_time_sec": round(sum(p["duration"] for p in pauses), 2),
    "voiced_ratio": round(voiced_ratio, 3),
    "arousal_frames_ratio": round(arousal, 3),
    "pauses_timeline": pauses,
    "words_per_minute": transcript_stats["words_per_minute"],
}
print(json.dumps(acoustic_profile, indent=2, ensure_ascii=False))
print(f"[OK] Acoustic profile ready. Pitch std (expressiveness) = "
      f"{acoustic_profile['pitch_std_hz']} Hz; pauses = "
      f"{acoustic_profile['significant_pauses_count']}")

In [ ]:
# -------------------------------------------------------------------------
# CELL 7 - Multilingual embedding of every utterance (target: R^768)
# -------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer


def build_embedder():
    try:
        m = SentenceTransformer(EMBEDDING_MODEL_ID, device="cuda")
        print("[i] Embedder:", EMBEDDING_MODEL_ID,
              "| dim:", m.get_sentence_embedding_dimension())
        return m
    except Exception as exc:
        print("[!] e5 unavailable -> MiniLM fallback:", exc)
        m = SentenceTransformer(EMBEDDING_FALLBACK_ID, device="cuda")
        print("[i] Embedder:", EMBEDDING_FALLBACK_ID,
              "| dim:", m.get_sentence_embedding_dimension())
        return m


embedder = build_embedder()
seg_src = [t for t in transcript_data if len(t["text"].split()) >= 1]
seg_ids = [f"SRG{i:03d}" for i in range(len(seg_src))]

seg_texts = [t["text"] for t in seg_src]
seg_meta = [
    {"seg_id": seg_ids[i], "start": seg_src[i]["start"], "end": seg_src[i]["end"],
     "lang": seg_src[i]["lang"], "words": len(seg_src[i]["text"].split()),
     "text": seg_src[i]["text"]}
    for i in range(len(seg_src))
]

vecs = embedder.encode(seg_texts, batch_size=16, show_progress_bar=True,
                       normalize_embeddings=True, convert_to_numpy=True)
vecs = vecs.astype(np.float32)
EMB_DIM = int(vecs.shape[1])

doc_vec = vecs.mean(axis=0)
doc_vec = doc_vec / np.linalg.norm(doc_vec)
doc_vec = doc_vec.astype(np.float32)

# --- Anchor claim vocabulary for GMM/MC (thesis ground-truth lexicon) ------
claim_keys = [
    "diverse peer voices reduce echo chambers",
    "familiar social voices feel authoritative to students",
    "short structured segments fit the commute window",
    "calm low-pressure audio suits fatigue after a long day",
    "mandatory co-hosting lowers the fear of solo broadcasting",
    "students practise privately but rarely publish",
    "university ID gate builds a closed trusted community",
    "peer validation builds credibility faster than ratings alone",
    "verification habits improve source trust",
    "students verify news before sharing it",
    "engagement concentration harms diversity of voices",
    "bounded attention during Uber rides limits podcast length",
    "podcast paradox: private recording, public silence",
    "Boost invitations encourage participation",
    "broadcaster reputation score drives listener trust",
    "corrections and transparency build credibility",
    "students value practical career and side-hustle topics",
    "student agency in shaping the broadcast",
    "broadcast clarity and structure matter",
    "interface simplicity and one-tap controls matter",
    "respectful moderation builds a safe room",
    "students listen but avoid creating alone",
    "thermodynamic constraint on attention and diversity",
    "epistemic agency of the student listener",
]
claim_emb = embedder.encode(claim_keys, batch_size=16, show_progress_bar=False,
                            normalize_embeddings=True, convert_to_numpy=True)
claim_emb = claim_emb.astype(np.float32)
claim_meta = {"claim_keys": claim_keys, "dim": int(claim_emb.shape[1])}

np.save(os.path.join(OUTPUT_DIR, "segment_embeddings.npy"), vecs)
np.save(os.path.join(OUTPUT_DIR, "full_transcript_embedding.npy"), doc_vec)
np.save(os.path.join(OUTPUT_DIR, "claim_book_embeddings.npy"), claim_emb)
with open(os.path.join(OUTPUT_DIR, "segment_meta.json"), "w", encoding="utf-8") as f:
    json.dump({"emb_dim": EMB_DIM, "n_segments": len(seg_meta), "segments": seg_meta},
              f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, "claim_book_meta.json"), "w", encoding="utf-8") as f:
    json.dump(claim_meta, f, ensure_ascii=False, indent=2)

print(f"[OK] Saved {vecs.shape[0]} segment embeddings ({EMB_DIM}-d) + full-doc + claim book.")
print(f"[OK] Full-transcript embedding head: {doc_vec[:8].round(3)}")

del embedder, vecs, claim_emb
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# -------------------------------------------------------------------------
# CELL 8 - OPTIONAL triangulation sources (Uber corpus / FGD export / survey)
# -------------------------------------------------------------------------
def sniff(path, cap=2400):
    try:
        if path.endswith(".jsonl") or path.endswith(".ndjson"):
            with open(path, encoding="utf-8", errors="replace") as f:
                text = f.read()[:cap]
        elif path.endswith(".json") or "_parts.json" in path:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
            text = json.dumps(data, ensure_ascii=False)[:cap]
        elif path.endswith(".csv"):
            with open(path, encoding="utf-8", errors="replace") as f:
                text = f.read()[:cap]
        elif os.path.isdir(path):
            files = [os.path.join(path, x) for x in sorted(os.listdir(path))[:6]]
            chunks = []
            for p in files:
                if not os.path.isdir(p):
                    with open(p, encoding="utf-8", errors="replace") as h:
                        chunks.append(p + " :: " + h.read()[:400])
                else:
                    chunks.append(p + " :: <dir>")
            text = "\n".join(chunks)
        else:
            with open(path, encoding="utf-8", errors="replace") as f:
                text = f.read()[:cap]
        return path, text
    except Exception as exc:
        return path, f"[unreadable: {exc}]"


extra_sources = []
for p in EXISTING_EXTRA:
    path, text = sniff(p)
    extra_sources.append({"path": path, "preview": text})
    print(f"[i] Ingested {os.path.basename(path)} ({len(text)} chars)")
print(f"[OK] {len(extra_sources)} triangulation source(s) ready (0 is fine).")

In [ ]:
# -------------------------------------------------------------------------
# CELL 9 - Load Qwen2.5 for deep reasoning (4-bit NF4, VRAM/RAM safe)
# -------------------------------------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_llm(model_id):
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokz = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    return tokz, mdl


torch.cuda.empty_cache()
print(f"[i] Loading {LLM_MODEL_ID} (4-bit)...")
LOADED_MODEL_ID = LLM_MODEL_ID
try:
    tok, llm = load_llm(LLM_MODEL_ID)
except Exception as exc:
    print(f"[!] {LLM_MODEL_ID} failed ({type(exc).__name__}); falling back to {LLM_FALLBACK_ID}.")
    tok, llm = load_llm(LLM_FALLBACK_ID)
    LOADED_MODEL_ID = LLM_FALLBACK_ID

print(f"[OK] LLM ready: {LOADED_MODEL_ID} | vocab {tok.vocab_size}")


def free_llm():
    global llm, tok
    del llm, tok
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# -------------------------------------------------------------------------
# CELL 10 - Thesis DNA injection (grounds every LLM answer in the framework)
# -------------------------------------------------------------------------
THESIS_CONTEXT = """
You are analysing a researcher self-report audio for a Master's thesis in Media Innovations (ACU).

RESEARCHER: Sherif Mohamed Abd Elmonem Rashwan, Teaching Assistant in Radio & TV. He recorded his
own account of how university students actually behave: their commutes (Uber rides in October 6th
City / Giza), media habits, trust and verification behaviour, fatigue, and whether they dare to speak.
This self-report is one GROUND-TRUTH source, to be triangulated with (a) Uber-rider participant
records, (b) the 52-record survey corpus, and (c) the focus-group discussion.

THESIS TITLE: 'The Social Media Radio Innovation and Its Impact on University Students' Ratings'.

CORE FRAMEWORK - EPISTEMIC SYSTEMS THEORY (EST) as proposed by the author:
- EST-P1 (Structural Determination): platform architecture determines epistemic outcomes.
- EST-P2 (Thermodynamic Constraint): diversity constraints vs. engagement concentration; echo
  chambers are traded against attention; the MORE algorithm and Echo Meter / Algorithmic Cooling
  instantiate this counterforce.
- EST-P3 (Psychometric Accountability): consequential metrics (Broadcaster Reputation Score, BRS)
  must be psychometrically accountable; trust and reputation are measured, not assumed.
- EST-P4 (Participation & Evidence-Based Production): student agency, media literacy, private
  practice vs. public broadcast - including the 'Podcast Paradox' (students record privately but
  fear public solo broadcasting).

PLATFORM: LOKLOK (1st Social Media Radio). Features: mandatory co-hosting, University-ID closed
ecosystem, 'Boost' invitations, Broadcaster Reputation Score (BRS: broadcast rigor + social
validation + information quality), MORE recommendation (value x diversity), Echo Meter EMA +
Algorithmic Cooling at R >= 0.75.

QUALITATIVE CONTEXT from the thesis's own coding:
- Rider-corpus themes T1-T6: (T1) accountable practice and social endorsement, (T2) familiar peer
  voices and socially local radio, (T3) a worthwhile ride need not be interactive, (T4) agency,
  contribution and protected access, (T5) relevance grounded in student lived needs, (T6) delivery
  and interface as enabling conditions.
- FGD domains: (FGD-T1) contextual listening and selective publication, (FGD-T2) collaborative
  broadcasting and choice, (FGD-T3) institutional eligibility / trust / privacy, (FGD-T4) interface
  cues and idle content.

YOUR TASK: act as a rigorous qualitative analyst and theoretical mapper. Ground every claim in the
transcript with its timestamp [XXs]. Be comprehensive, elaborative, critical, and honest about the
difference between observation, interpretation, and design implication. Never invent quotations
that are not in the transcript. Think step-by-step and be exceptionally specific.
"""

print("[OK] Thesis DNA loaded (", len(THESIS_CONTEXT), "chars).")

In [ ]:
# -------------------------------------------------------------------------
# CELL 11 - Reasoning engine + transcript chunking (CoT-style, memory-safe)
# -------------------------------------------------------------------------
import re as _re


def deep_reasoning_pass(prompt_text, system=None, max_tokens=MAX_CHAPTER_TOKENS,
                        temperature=0.6):
    sys_msg = system or THESIS_CONTEXT
    msgs = [
        {"role": "system", "content": sys_msg + "\n\nThink step-by-step. Be highly analytical, "
                                               "critical, and elaborate. Quote transcript timestamps."},
        {"role": "user", "content": prompt_text},
    ]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    if isinstance(enc, dict):
        ids = enc["input_ids"]
    else:
        ids = enc
    ids = ids.to(llm.device)
    attn = torch.ones_like(ids)
    with torch.no_grad():
        gen = llm.generate(
            input_ids=ids,
            attention_mask=attn,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tok.eos_token_id,
        )
    out = tok.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip()
    del gen, ids
    gc.collect()
    torch.cuda.empty_cache()
    return out


def make_chunks(meta_list, budget=CHUNK_CHAR_BUDGET):
    chunks, cur = [], []
    for m in meta_list:
        line = f"[{m['start']:.1f}s-{m['end']:.1f}s|{m['lang']}] {m['text']}"
        if cur and sum(len(x) for x in cur) + len(line) > budget:
            chunks.append("\n".join(cur))
            cur = []
        cur.append(line)
    if cur:
        chunks.append("\n".join(cur))
    return chunks


chunks = make_chunks(seg_meta)
print(f"[OK] {len(chunks)} transcript chunk(s) for the reasoning engine.")

In [ ]:
# -------------------------------------------------------------------------
# CELL 12 - Phases 1-3 of reflexive thematic analysis: familiarisation +
# open coding of every chunk (Braun & Clarke)
# -------------------------------------------------------------------------
def open_code_chunk(chunk, idx):
    return deep_reasoning_pass(f'''
You are performing REFLEXIVE THEMATIC ANALYSIS (Braun & Clarke phases 1-3):
familiarisation, generating initial codes, open coding - one code per utterance/claim.

TRANSCRIPT CHUNK {idx} of {len(chunks)}:
{chunk}

Return a markdown table with EXACTLY this structure (no prose before/after):
| code_id | code_label | start_s | quote_original | gloss_en | analytic_memo | est_hint |

Rules:
- code_id: sequential within each chunk using the chunk prefix, e.g. C{idx:02d}-01, C{idx:02d}-02, ...
- quote_original: the shortest exact utterance including its [timestamp], keeping Arabic where spoken.
- gloss_en: concise English gloss of the quoted utterance.
- analytic_memo: 1-2 sentences on what this reveals about the student as an EPISTEMIC AGENT
  (commuting, fatigued, verifying, trusting peers, choosing silence vs voice).
- est_hint: map to EST-P1/P2/P3/P4 and/or Podcast Paradox / BRS / MORE / co-hosting /
  University-ID gate when applicable; else 'none'.
- Be exhaustive: cover EVERY distinct idea in the chunk.
''')


codes_raw = []
for ci, chunk in enumerate(chunks):
    print(f"  Phase 1-3 | chunk {ci + 1}/{len(chunks)}")
    txt = open_code_chunk(chunk, ci + 1)
    codes_raw.append({"chunk_index": ci, "text": txt})

CODES_GLOSS = "\n\n".join(c["text"] for c in codes_raw)
with open(os.path.join(OUTPUT_DIR, "thematic_codes_raw.txt"), "w", encoding="utf-8") as f:
    f.write(CODES_GLOSS)
print(f"[OK] Open codes generated ({len(codes_raw)} chunks, ~{len(CODES_GLOSS)} chars).")

In [ ]:
# -------------------------------------------------------------------------
# CELL 13 - Phases 4-5: search, review, define & NAME the themes (Braun & Clarke)
# -------------------------------------------------------------------------
def extract_json_block(text):
    text = text.strip()
    if text.startswith("```"):
        text = _re.sub(r"^```[A-Za-z]*\s*", "", text)
        text = _re.sub(r"\s*```$", "", text)
    s, e = text.find("{"), text.rfind("}")
    if s >= 0 and e > s:
        return json.loads(text[s:e + 1])
    raise ValueError("No JSON object found in model output.")


theme_prompt = f'''
You have performed open coding of a researcher self-report transcript about university student
behaviour (a ground-truth source for an EST / LOKLOK thesis). Codes (each with code id, label,
timestamped quotation, analytic memo, est_hint):

---CODES---
{CODES_GLOSS}

---ACOUSTIC CONTEXT---
Pitch mean {acoustic_profile['pitch_mean_hz']} Hz (std {acoustic_profile['pitch_std_hz']} Hz,
expressive), energy range {acoustic_profile['energy_range_db']} dB,
{acoustic_profile['significant_pauses_count']} pauses,
{acoustic_profile['words_per_minute']} wpm, duration {acoustic_profile['duration_sec']} s.

Now do Braun & Clarke PHASES 4-5 (searching for themes, reviewing them, defining & naming them).
1. Group the codes into coherent candidate themes.
2. Crosswalk each candidate theme to the thesis's existing qualitative landmarks: T1-T6 (rider
   corpus) and/or FGD-T1..T4 and/or an EST proposition; OR explicitly mark it as NEW (id "SRT#")
   if it is genuinely new.
3. Review every theme against the codes and the transcript evidence; prune or merge weak ones.
4. Define and name every theme precisely: definition includes scope, what counts as evidence, and the
   boundary of the claim.

Return ONLY a single JSON object (valid JSON, no markdown) with this shape:
{{
  "themes": [
    {{
      "id": "T1",
      "name": "short theme name",
      "definition": "comprehensive definition (scope + evidence + boundary)",
      "codes": ["C01-01", "C01-03"],
      "representative_quote": "[timestamp] exact quote (Arabic + English gloss)",
      "rider_corpus_crosswalk": "T1-T6 or null",
      "fgd_crosswalk": "FGD-Tx or null",
      "est_propositions": ["EST-P1"],
      "design_implication": "concrete LOKLOK design consequence"
    }}
  ]
}}
'''

print("  Phase 4-5 | theme development")
themes_obj = None
for _attempt in range(3):
    themes_out = deep_reasoning_pass(theme_prompt, max_tokens=2000,
                                     temperature=0.4 if _attempt else 0.6)
    try:
        themes_obj = extract_json_block(themes_out)
        break
    except Exception as _exc:
        print(f"    [retry {_attempt + 1}/3] theme JSON parse failed: {_exc}")
        print("    model head:", themes_out[:300].replace(chr(10), " "))
if themes_obj is None:
    raise RuntimeError("Theme analysis produced no valid JSON after 3 attempts.")
themes = themes_obj["themes"]
print(f"[OK] {len(themes)} themes defined.")
for th in themes:
    print(f"   - {th.get('id', '?')} | {th.get('name', '?')}")
with open(os.path.join(OUTPUT_DIR, "themes_defined.json"), "w", encoding="utf-8") as f:
    json.dump(themes, f, ensure_ascii=False, indent=2)

In [ ]:
# -------------------------------------------------------------------------
# CELL 14 - Pass A: behavioural & narrative synthesis (ground-truth patterns)
# -------------------------------------------------------------------------
behavior_pass = deep_reasoning_pass(f'''
You are synthesising the SELF-REPORT GROUND TRUTH for the thesis.

THEMES DEVELOPED:
{json.dumps(themes, ensure_ascii=False, indent=2)[:4500]}

TIMESTAMPED TRANSCRIPT (concise view):
{TS_TEXT[:9000]}

Write a comprehensive, elaborative synthesis (structured markdown) covering:
1. SPECIFIC STUDENT BEHAVIOURS: daily routines, commutes (Uber rides), fatigue, attention windows,
   media and verification habits, trust in peers vs institutions, reasons for silence.
2. STORYTELLING / ANECDOTES: the concrete stories told and what they reveal about the 'Student as
   Epistemic Agent'.
3. THE PODCAST PARADOX & PARTICIPATION: evidence of students wanting to speak but holding back,
   private practice vs public broadcasting, peer-to-peer preference.
4. Implied MEASUREMENT constraints for the BRS and for the MORE diversity engine.

Every behavioural claim MUST cite a [timestamp]. Quote original Arabic phrases and give an English
gloss. Distinguish what is directly observed, what is inferred, and what remains uncertain.
''')
print("[OK] Behaviours synthesised (", len(behavior_pass), "chars).")

In [ ]:
# -------------------------------------------------------------------------
# CELL 15 - Pass B: acoustic-semantic reflexivity & positionality
# -------------------------------------------------------------------------
reflexivity_pass = deep_reasoning_pass(f'''
You are analysing the RESEARCHER'S vocal delivery and its relation to the content he is narrating,
in order to write the REFLEXIVITY section of the ground-truth report.

ACOUSTIC DATA:
- Mean pitch: {acoustic_profile['pitch_mean_hz']} Hz (male-voice narration baseline ~100-140 Hz)
- Pitch std (expressiveness): {acoustic_profile['pitch_std_hz']} Hz
- Energy dynamic range: {acoustic_profile['energy_range_db']} dB
- Significant reflection pauses: {acoustic_profile['significant_pauses_count']}
  (total {acoustic_profile['total_reflection_time_sec']} s)
- Speech tempo: {acoustic_profile['tempo_bpm']} bpm | {acoustic_profile['words_per_minute']} wpm
- Voiced ratio: {acoustic_profile['voiced_ratio']} | arousal frames {acoustic_profile['arousal_frames_ratio']}

NARRATIVE SUMMARY (short):
{behavior_pass[:2000]}

Write a comprehensive, elaborative REFLEXIVITY & POSITIONALITY analysis:
1. EMOTIONAL / COGNITIVE MARKERS: where does the researcher's voice likely convey fatigue, empathy,
   frustration, or realisation? Connect pitch/energy/pauses to narrative moments (e.g. 'the elongated
   pause before [timestamp] while discussing student fatigue suggests ...'). Be disciplined: prosody
   differs from semantics; mark inference as inference.
2. POSITIONALITY: the researcher is a TEACHING ASSISTANT = institutional insider yet not a student.
   Discuss insider/outsider dynamics, the power relationship, and how this shapes what he can and
   cannot see.
3. BIAS CHECK: what preconceived notions about student media literacy might the storytelling reveal?
   How must EST-P4's evidence-based-production interpretation be guarded against them?
4. REFLEXIVITY PROTOCOL: 3-5 concrete steps to keep this self-report honest as ground truth
   (e.g. member-checks, triangulation rules, coding-decision logs).
''')
print("[OK] Reflexivity analysis done (", len(reflexivity_pass), "chars).")

In [ ]:
# -------------------------------------------------------------------------
# CELL 16 - Pass C: formal EST mapping + LOKLOK design ground-truth rules
# -------------------------------------------------------------------------
extra_text = "\n\n".join(
    s["path"] + "\n" + s["preview"] for s in extra_sources
) if extra_sources else "NO EXTRA SOURCES SUPPLIED."

est_pass = deep_reasoning_pass(f'''
Synthesise the self-report ground truth into a FORMAL mapping for the thesis, integrating the
triangulation sources where they exist.

BEHAVIOURAL SYNTHESIS:
{behavior_pass[:2500]}

REFLEXIVITY:
{reflexivity_pass[:1200]}

TRIANGULATION MATERIAL (optional, may be empty):
{extra_text}

Deliver comprehensive, elaborative markdown with EXACTLY these sections:
1. EST-P1 (Structural Determination): how observed daily routines (commute fatigue, bounded
   attention, passenger posture) dictate LOKLOK's structural design (broadcast length, session
   scheduling, radio vs feed).
2. EST-P2 (Diversity / Thermodynamic constraint): do the stories show echo chambers or a desire
   for diverse peer voices? Justify the MORE algorithm and Echo Meter / Cooling thresholds from
   observed reality.
3. EST-P3 (Psychometric accountability): what does the report imply the BRS must and must NOT
   claim? Feedback loops between reputation, attention, and trust; measurement accountability
   constraints.
4. EST-P4 (Participation / Podcast Paradox): how does the Paradox manifest in these specific
   stories, and how do MANDATORY CO-HOSTING + University-ID gate + Boost invitations respond?
5. GROUND-TRUTH RULES: exactly 3-5 definitive, falsifiable design rules for BRS/UI derived
   STRICTLY from these observations (each with [timestamp evidence] and a success metric to test it).
6. TRIANGULATION READINESS: a table mapping this self-report's themes to (T1-T6 rider corpus,
   FGD-T1-T4, survey) as convergent / divergent / complementary, using the extra material when
   present and marking gaps.
''', max_tokens=2200)
print("[OK] EST mapping + design rules done (", len(est_pass), "chars).")

In [ ]:
# -------------------------------------------------------------------------
# CELL 17 - GMM readiness: latent structure of the segment embeddings
# -------------------------------------------------------------------------
import matplotlib
import matplotlib.pyplot as plt  # noqa: E402 (inline in Colab; the PNG is also saved)

from sklearn.decomposition import PCA  # noqa: E402
from sklearn.mixture import GaussianMixture  # noqa: E402

Xseg = np.load(os.path.join(OUTPUT_DIR, "segment_embeddings.npy"))
with open(os.path.join(OUTPUT_DIR, "segment_meta.json"), encoding="utf-8") as f:
    segmeta = json.load(f)
segs = segmeta["segments"]
assert len(segs) == Xseg.shape[0], "meta/embedding length mismatch"

n_seg, dim = Xseg.shape
if n_seg < 3:
    raise RuntimeError("Not enough segments for clustering (need at least 3).")

K_PCA = max(2, min(60, n_seg - 1))
pca = PCA(n_components=K_PCA, random_state=SEED)
Xp = pca.fit_transform(Xseg)
cum = np.cumsum(pca.explained_variance_ratio_)
print(f"[i] {n_seg} segments x {dim}-d -> PCA {K_PCA}-d, retained variance {cum[-1]:.3f}")

bic = {}
k_max = min(8, n_seg - 1)
for k in range(1, k_max + 1):
    m = GaussianMixture(n_components=k, covariance_type="diag",
                        random_state=SEED, max_iter=400).fit(Xp)
    bic[k] = m.bic(Xp)
best_k = min(bic, key=bic.get)
print("[i] BIC per k:", {k: round(v) for k, v in bic.items()}, "-> best k =", best_k)

gmm = GaussianMixture(n_components=best_k, covariance_type="diag",
                      random_state=SEED, max_iter=400)
cluster_labels = gmm.fit_predict(Xp)
probs = gmm.predict_proba(Xp)

cluster_table = []
for ci in range(best_k):
    idx = np.where(cluster_labels == ci)[0]
    top_idx = idx[np.argsort(-probs[idx, ci])[:5]]
    cluster_table.append({
        "cluster": int(ci),
        "size": int(idx.size),
        "share": round(float(idx.size / n_seg), 3),
        "top_segments": [{"seg_id": segs[j]["seg_id"], "start": segs[j]["start"],
                          "end": segs[j]["end"], "text": segs[j]["text"][:260]}
                         for j in top_idx],
    })

# PCA 2-D scatter (only for display)
pca2 = PCA(n_components=2, random_state=SEED)
X2 = pca2.fit_transform(Xseg)
plt.figure(figsize=(9, 6))
sc = plt.scatter(X2[:, 0], X2[:, 1], c=cluster_labels, cmap="tab10",
                 s=46, alpha=0.85, edgecolors="k", linewidths=0.4)
plt.colorbar(sc, ticks=range(best_k), label="GMM cluster")
plt.title(f"Segment embeddings (PCA 2-D) coloured by GMM cluster (k={best_k}, BIC-selected)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "gmm_clusters_pca.png")
plt.savefig(plot_path, dpi=150)
plt.show()

gmm_out = {"best_k": int(best_k),
           "bic": {str(k): float(v) for k, v in bic.items()},
           "pca_components_used": int(K_PCA),
           "exp_var_ratio": round(float(cum[-1]), 4),
           "cluster_table": cluster_table,
           "cluster_labels": [int(x) for x in cluster_labels],
           "membership_prob_max": [round(float(mp), 3) for mp in probs.max(axis=1)]}
with open(os.path.join(OUTPUT_DIR, "gmm_readiness.json"), "w", encoding="utf-8") as f:
    json.dump(gmm_out, f, ensure_ascii=False, indent=2)
print(f"[OK] GMM readiness saved (k={best_k}); plot -> {plot_path}")

In [ ]:
# -------------------------------------------------------------------------
# CELL 18 - Monte-Carlo readiness: bootstrap CI + Echo-Meter trigger sim
# -------------------------------------------------------------------------
claim_emb = np.load(os.path.join(OUTPUT_DIR, "claim_book_embeddings.npy"))
doc_vec = np.load(os.path.join(OUTPUT_DIR, "full_transcript_embedding.npy"))
rng = np.random.default_rng(SEED)


def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(a @ b)


def ci(arr, lo=2.5, hi=97.5):
    return [round(float(np.percentile(arr, lo)), 4), round(float(np.percentile(arr, hi)), 4)]


# 1) Cluster-share bootstrap (GMM labels computed in the previous cell)
B = 4000
labels = np.asarray(gmm_out["cluster_labels"])
n_seg_all = len(labels)
share_sims = {str(k): [] for k in range(best_k)}
for _ in range(B):
    idx = rng.integers(0, n_seg_all, size=n_seg_all)
    for k in range(best_k):
        share_sims[str(k)].append(float((labels[idx] == k).mean()))

mc_cluster = {"bootstrap_reps": int(B),
              "cluster": {str(k): {"mean_share": round(float(np.mean(share_sims[str(k)])), 4),
                                   "mc_se": round(float(np.std(share_sims[str(k)], ddof=1)), 4),
                                   "ci95": ci(share_sims[str(k)])}
                          for k in range(best_k)}}

# 2) Embedding diversity (1 - cosine = DBI) of each claim vs the document-level
#    'recent consumption' vector, then a Monte-Carlo Echo-Meter simulation.
recent = doc_vec
alpha_r = 0.30
dbi_items = {}
for i, key in enumerate(claim_meta["claim_keys"]):
    dbi_items[key] = round(1.0 - cosine_sim(claim_emb[i], recent), 4)

T = 40            # sessions simulated past the observation window
reps = 1500
R = np.zeros((reps, T + 1))
for r_i in range(reps):
    rval = 0.35
    R[r_i, 0] = rval
    draws = rng.integers(0, len(claim_emb), size=T)
    for t in range(1, T + 1):
        d = 1.0 - cosine_sim(claim_emb[draws[t - 1]], recent)
        rval = (1 - alpha_r) * rval + alpha_r * d
        R[r_i, t] = rval

mc_echo = {
    "sessions_simulated": int(T),
    "monte_carlo_reps": int(reps),
    "alpha_r": alpha_r,
    "final_R_mean": round(float(R[:, T].mean()), 4),
    "final_R_sd": round(float(R[:, T].std(ddof=1)), 4),
    "final_R_mc_se": round(float(R[:, T].std(ddof=1) / np.sqrt(reps)), 4),
    "p_trigger_0_60": round(float(np.mean(R[:, T] >= 0.60)), 4),
    "p_trigger_0_75": round(float(np.mean(R[:, T] >= 0.75)), 4),
    "threshold_breach_probabilities": {
        "0.60": [round(float(np.mean(R[:, tt] >= 0.60)), 3) for tt in (5, 10, 20, 30, 40)],
        "0.75": [round(float(np.mean(R[:, tt] >= 0.75)), 3) for tt in (5, 10, 20, 30, 40)],
    },
}

mc_out = {
    "bootstrap_reps": int(B),
    "cluster_bootstrap": mc_cluster,
    "dbi_vs_doc_claim_book": dbi_items,
    "echo_meter_simulation": mc_echo,
    "interpretation_note": (
        "Monte-Carlo objects here quantify READINESS and uncertainty for the thesis's planned "
        "G0/G1 replication and MORE/Echo-Meter design: bootstrap CIs bound the prevalence of "
        "latent clusters in THIS ground-truth corpus; the Echo-Meter sim reports how often the "
        "embedding-derived diversity trigger would fire (and the MC-SE of R), NOT human effects."
    ),
}
with open(os.path.join(OUTPUT_DIR, "monte_carlo_readiness.json"), "w", encoding="utf-8") as f:
    json.dump(mc_out, f, ensure_ascii=False, indent=2)

print("[OK] Monte-Carlo readiness saved.")
print("  cluster share MC-SE:", {k: v["mc_se"] for k, v in mc_cluster["cluster"].items()})
print("  Echo Meter R end: mean", mc_echo["final_R_mean"],
      "| MC-SE", mc_echo["final_R_mc_se"],
      "| P(R>=0.60)", mc_echo["p_trigger_0_60"],
      "| P(R>=0.75)", mc_echo["p_trigger_0_75"])

In [ ]:
# -------------------------------------------------------------------------
# CELL 19 - Compile the comprehensive LaTeX + Markdown ground-truth report
# -------------------------------------------------------------------------
def tex_escape(s):
    s = str(s)
    s = s.replace("\\", r"\textbackslash{}")
    for ch, rep in (("%", r"\%"), ("&", r"\&"), ("#", r"\#"), ("_", r"\_"),
                    ("$", r"\$"), ("~", r"\textasciitilde{}"), ("^", r"\textasciicircum{}"),
                    ("{", r"\{"), ("}", r"\}")):
        s = s.replace(ch, rep)
    return s


def short_text(s, n=200):
    return str(s).replace("\n", " ")[:n]


# --- LaTeX fragments -------------------------------------------------------
theme_rows = []
for t in themes:
    theme_rows.append(
        " & ".join([
            tex_escape(t.get("id", "?")),
            tex_escape(t.get("name", "?")),
            tex_escape(t.get("rider_corpus_crosswalk", "null")),
            tex_escape(t.get("fgd_crosswalk", "null")),
            tex_escape(short_text(t.get("definition", ""), 200)),
        ]) + " \\\\"
    )
theme_rows_tex = "\n".join(theme_rows)

cluster_rows = []
for c in gmm_out["cluster_table"]:
    top_txt = short_text(c["top_segments"][0]["text"], 140) if c["top_segments"] else "n/a"
    cluster_rows.append(
        " & ".join([str(c["cluster"]), str(c["size"]), str(c["share"]),
                    tex_escape(top_txt)]) + " \\\\"
    )
cluster_rows_tex = "\n".join(cluster_rows)

mc_rows = []
for k, v in mc_cluster["cluster"].items():
    mc_rows.append(" & ".join([str(k), str(gmm_out["cluster_table"][int(k)]["size"]),
                               str(v["mean_share"]), str(v["mc_se"]),
                               str(v["ci95"][0]) + "--" + str(v["ci95"][1])]) + " \\\\")
mc_rows_tex = "\n".join(mc_rows)

extra_items = "\n".join(r"\item " + tex_escape(s["path"]) for s in extra_sources)
if not extra_items:
    extra_items = (r"\item None supplied in this run -- triangulation with the Uber corpus, "
                   r"survey and FGD is reserved as the next step.")

TRIANG_TEXT = (
    "The four strands are convergent where students consistently describe bounded attention during "
    "commutes, socially proximate trust, and private practice without public publishing; they are "
    "divergent where the teaching-assistant perspective may over-weight institutional-membership "
    "benefits relative to the rider accounts; they are complementary where acoustic-reflexive and "
    "embedding data add delivery-level evidence the other strands cannot provide. Each divergence "
    "is retained as an open audit item rather than smoothed away."
)

# --- LaTeX document ---------------------------------------------------------
L = []
L.append(r"""\section{Ground-Truth Researcher Self-Report: Thematic Analysis and Epistemic Mapping}
\label{sec:self_report_ground_truth}

\subsection{Role of this Source in the Mixed-Methods Design}
This appendix records a \textit{researcher self-report}: the teaching assistant's account of how
university students behave, commute, consume media, verify information, and decide whether to speak.
It is one of four triangulated ground-truth strands in this thesis: (a) the Uber-rider participant
records, (b) the 52-record survey corpus, (c) the focus-group discussion, and (d) this self-report.
As reflexive evidence it does not stand alone; every claim is cross-examined against the other
strands and against the quantitative model. The recording was produced by the researcher himself,
which makes his positionality an explicit analytical object rather than an implicit bias.
""")

L.append(r"""\subsection{Recording, Transcription, and Acoustic Profiling}
""")
L.append(
    "Audio source: " + tex_escape(AUDIO_PATH) +
    " (" + str(acoustic_profile['duration_sec']) + " s). " +
    "Transcribed with Whisper Large-V3 (VAD, word timestamps, automatic language detection): " +
    str(transcript_stats['speech_duration_sec']) + " s of speech, " +
    str(transcript_stats['segments']) + " segments, " +
    str(transcript_stats['words']) + " words at " +
    str(transcript_stats['words_per_minute']) + " words/min. Language mix: " +
    tex_escape(json.dumps(transcript_stats['language_segments'], ensure_ascii=False)) + "."
)
L.append(
    "\n\nAcoustic/paralinguistic profile (the ``how'' layer of reflexivity): mean pitch " +
    str(acoustic_profile['pitch_mean_hz']) + "\\,Hz (male-voice narration baseline), " +
    "pitch variance " + str(acoustic_profile['pitch_std_hz']) + "\\,Hz (expressiveness), " +
    "energy dynamic range " + str(acoustic_profile['energy_range_db']) + "\\,dB, tempo " +
    str(acoustic_profile['tempo_bpm']) + "\\,bpm, voiced ratio " +
    str(acoustic_profile['voiced_ratio']) + ", significant pauses " +
    str(acoustic_profile['significant_pauses_count']) + " (total reflection time " +
    str(acoustic_profile['total_reflection_time_sec']) + "\\,s), arousal frames " +
    str(acoustic_profile['arousal_frames_ratio']) + "."
)
L.append(
    "\n\nHigh pitch variance and clustered pauses indicate engaged, reflective narration rather" +
    " than rehearsed recitation; this proxies cognitive involvement at moments such as the" +
    " discussion of student fatigue during Uber rides. Prosodic interpretation remains" +
    " inferential and is flagged as such. All recorded codes, timestamps and raw acoustic arrays" +
    " are archived beside this appendix."
)

L.append(r"""\subsection{Reflexive Thematic Analysis (Braun \& Clarke, Six Phases)}
Analysis followed the six phases: (1) familiarisation (multilingual transcription plus acoustic
listening); (2) generating initial codes (open coding of every chunk; codes Cxx-yy); (3) searching
for themes; (4) reviewing themes against codes and evidence; (5) defining and naming final themes;
(6) producing this report. The codebook below lists the final themes, their crosswalk to the rider
corpus themes (T1--T6), the focus-group domains (FGD-T1--T4), and the EST propositions they most
directly inform.
""")

L.append(r"""\begin{longtable}{@{}llllp{6.3cm}@{}}
\caption{Final Themes of the Researcher Self-Report (Ground-Truth Codebook)}
\label{tab:sr-themes}\\
\toprule
ID & Theme & Rider ref & FGD ref & Definition / Evidence \\
\midrule
\endfirsthead
\multicolumn{5}{l}{\small\itshape Table~\thetable\ continued}\\
\toprule
ID & Theme & Rider ref & FGD ref & Definition / Evidence \\
\midrule
\endhead
""")
L.append(theme_rows_tex)
L.append(r"""\bottomrule
\end{longtable}
\tabnote{Evidence is timestamped self-report material; definitions state scope, evidence and
boundary. Representative quoted evidence for each theme appears below.}
\FloatBarrier
""")

L.append(r"""\subsection{Theme Descriptions, Evidence, and Reflexive Boundaries}
""")
for t in themes:
    L.append(
        "\n\\textbf{" + tex_escape(t.get("id", "?")) + " -- " +
        tex_escape(t.get("name", "?")) + ".} " +
        tex_escape(t.get("definition", ""))
    )
    if t.get("representative_quote"):
        L.append(" Representative quote: \\textit{" +
                 tex_escape(t.get("representative_quote", "")) + "}.")
    L.append(" \\\\\n\\textit{LOKLOK consequence:} " +
             tex_escape(t.get("design_implication", "n/a")) + "\n")

L.append(r"""\subsection{Behavioural Synthesis and the Podcast Paradox (EST-P4)}
""")
L.append(tex_escape(behavior_pass))

L.append(r"""\subsection{Researcher Reflexivity and Positionality}
""")
L.append(tex_escape(reflexivity_pass))

L.append(r"""\subsection{Formal Ground-Truth Mapping to EST and LOKLOK Design}
""")
L.append(tex_escape(est_pass))

L.append(r"""\subsection{Computational Grounding: Embeddings, GMM, and Monte-Carlo Readiness}
Utterance-level multilingual embeddings were produced with a multilingual sentence encoder
""")
L.append(
    "(" + str(EMB_DIM) + "-dimensional, matching the thesis domain model " +
    "$\\mathbf{h}_s\\in\\mathbb{R}^{768}$ when the e5 model is used) and clustered with a " +
    "BIC-selected Gaussian Mixture over " + str(gmm_out["pca_components_used"]) +
    " PCA components (k = " + str(gmm_out["best_k"]) + "; retained variance " +
    str(gmm_out["exp_var_ratio"]) + "). The full-document embedding head is " +
    str(np.round(doc_vec[:6].astype(float), 3).tolist()) + "."
)
L.append(
    "\n\nEach GMM cluster groups narrative episodes by epistemic stance; cluster shares carry" +
    " bootstrap uncertainty (Monte-Carlo standard errors below). These objects feed the reserved" +
    " G0/G1 replication tables and the MORE / Echo-Meter design simulation."
)

L.append(r"""\begin{longtable}{@{}crrcr@{}}
\caption{Monte-Carlo Readiness: GMM Cluster Shares with Bootstrap Intervals ($B=4000$)}
\label{tab:sr-mc}\\
\toprule
Cluster & n & Share & MC-SE & 95\% CI \\
\midrule
\endfirsthead
\multicolumn{5}{l}{\small\itshape Table~\thetable\ continued}\\
\toprule
Cluster & n & Share & MC-SE & 95\% CI \\
\midrule
\endhead
""")
L.append(mc_rows_tex)
L.append(r"""\bottomrule
\end{longtable}
\tabnote{Intervals are 4{,}000 bootstrap resamples of segment membership; they quantify sampling
uncertainty of the observed ground-truth distribution, not population prevalence.}
\FloatBarrier
""")

L.append(
    "\\textbf{Echo-Meter / MORE readiness (embedding-driven Monte-Carlo).} Over " +
    str(mc_echo["monte_carlo_reps"]) + " simulated 40-session traces, the diversity trigger R ended" +
    " at mean " + str(mc_echo["final_R_mean"]) + " (MC-SE " + str(mc_echo["final_R_mc_se"]) +
    "); P(R$\\geq$0.60) = " + str(mc_echo["p_trigger_0_60"]) + " and P(R$\\geq$0.75) = " +
    str(mc_echo["p_trigger_0_75"]) + "."
)
L.append(
    "\n\nThese probabilities bound how often the proposed MORE/Echo-Meter counterforce would engage" +
    " given the claim-book items under resampled exposure order. They are design-sensitivity" +
    " numbers for the prospective experiment, not measured human behaviour."
)

L.append(r"""\subsection{Triangulation with the Other Ground-Truth Strands}
Sources ingested and reviewed in this run:
\begin{itemize}
""")
L.append(extra_items)
L.append(r"""\end{itemize}
""")
L.append(TRIANG_TEXT)

L.append(r"""
\subsection{Trustworthiness, Ethics, and Limitations}
Trustworthiness follows Lincoln \& Guba: credibility (multilingual transcription, timestamped
evidence, reflexive audit trail), transferability (thick context: Giza/October 6th City commutes,
ACU setting), dependability (open codebook, coding-decision log archived beside the transcript),
and confirmability (acoustic arrays and embeddings archived as independent artefacts; inferences
are labelled). Ethics: the self-report is the researcher's own narration; no student identifiers
were solicited or stored. Limitations: single-source reflexivity, prosodic inference, ASR error on
strong dialect, no member-check yet, and GMM/Monte-Carlo objects are readiness artefacts, not
evidence of causal effects. This material does not by itself validate an EST proposition or a BRS
design choice; it grounds them for subsequent experimental test.

\vspace{0.4em}
\noindent\textit{Source chain:} audio $\to$ Whisper transcription $\to$ librosa acoustics $\to$
multilingual embeddings ($\mathbb{R}^{768}$) $\to$ Qwen 2.5 thematic/deep passes $\to$ GMM +
Monte-Carlo readiness $\to$ this report.
""")

latex_report = "\n".join(L)

with open(os.path.join(OUTPUT_DIR, "thesis_self_report_ground_truth.tex"), "w", encoding="utf-8") as f:
    f.write(latex_report)

# --- Markdown mirror --------------------------------------------------------
M = []
M.append("# Ground-Truth Researcher Self-Report: Thematic Analysis and EST Mapping\n\n")
M.append(f"**Audio:** {AUDIO_PATH} ({acoustic_profile['duration_sec']} s)  \n")
M.append(f"**Whisper Large-V3:** {transcript_stats['segments']} segments, "
         f"{transcript_stats['words']} words, {transcript_stats['words_per_minute']} wpm  \n")
M.append(f"**Language mix:** {json.dumps(transcript_stats['language_segments'], ensure_ascii=False)}\n\n")
M.append("## 1. Acoustic / Paralinguistic Profile\n\n")
M.append(f"- Pitch mean {acoustic_profile['pitch_mean_hz']} Hz; pitch std (expressiveness) "
         f"{acoustic_profile['pitch_std_hz']} Hz\n")
M.append(f"- Energy range {acoustic_profile['energy_range_db']} dB; tempo "
         f"{acoustic_profile['tempo_bpm']} bpm; voiced ratio {acoustic_profile['voiced_ratio']}; "
         f"arousal frames {acoustic_profile['arousal_frames_ratio']}\n")
M.append(f"- Significant pauses {acoustic_profile['significant_pauses_count']} "
         f"(total {acoustic_profile['total_reflection_time_sec']} s)\n\n")
M.append("## 2. Reflexive Thematic Analysis (Braun & Clarke, 6 phases)\n\n")
M.append("| Theme id | Name | Rider ref | FGD ref | Definition | Representative quote |\n")
M.append("| --- | --- | --- | --- | --- | --- |\n")
for t in themes:
    M.append(f"| {t.get('id', '')} | {t.get('name', '').replace('|', '/')} | "
             f"{t.get('rider_corpus_crosswalk', '')} | {t.get('fgd_crosswalk', '')} | "
             f"{short_text(t.get('definition', ''), 200)} | "
             f"{short_text(t.get('representative_quote', ''), 160).replace('|', '/')} |\n")
M.append("\n### Behavioral & Narrative Synthesis\n\n")
M.append(str(behavior_pass)[:50000] + "\n\n")
M.append("### Reflexivity & Positionality (Pass B)\n\n")
M.append(str(reflexivity_pass)[:50000] + "\n\n")
M.append("### EST Mapping & LOKLOK Ground-Truth Rules (Pass C)\n\n")
M.append(str(est_pass)[:50000] + "\n\n")
M.append("### GMM + Monte-Carlo Readiness\n\n")
M.append(f"- Best GMM k = {gmm_out['best_k']} (BIC); PCA {gmm_out['pca_components_used']} comps "
         f"({gmm_out['exp_var_ratio']} retained variance)\n")
M.append("| Cluster | size | share | MC-SE | 95% CI |\n|---|---|---|---|---|\n")
for k, v in mc_cluster["cluster"].items():
    M.append(f"| {k} | {gmm_out['cluster_table'][int(k)]['size']} | {v['mean_share']} | "
             f"{v['mc_se']} | {v['ci95'][0]}--{v['ci95'][1]} |\n")
M.append(f"- Echo-Meter MC: mean R = {mc_echo['final_R_mean']}, MC-SE = {mc_echo['final_R_mc_se']}, "
         f"P(R>=0.60) = {mc_echo['p_trigger_0_60']}, P(R>=0.75) = {mc_echo['p_trigger_0_75']}\n\n")
M.append("## 3. Triangulation, Trustworthiness, Limitations\n\n")
M.append("Sources ingested for triangulation: " +
         (", ".join(os.path.basename(s["path"]) for s in extra_sources) if extra_sources
          else "none in this run") + ".\n\n")
M.append(TRIANG_TEXT + "\n\n")
M.append("*GMM/Monte-Carlo objects are **readiness** artifacts; they are not claims of causal or "
         "population effects.*\n")

md_report = "".join(M)
with open(os.path.join(OUTPUT_DIR, "thesis_self_report_ground_truth.md"), "w", encoding="utf-8") as f:
    f.write(md_report)

# --- JSON archive ------------------------------------------------------------
archive = {
    "meta": {"llm_model": LOADED_MODEL_ID, "embedding_model": EMBEDDING_MODEL_ID,
             "embedding_dim": int(EMB_DIM), "seed": int(SEED)},
    "transcript_stats": transcript_stats,
    "acoustic_profile": acoustic_profile,
    "themes": themes,
    "codes_raw_chunks": len(codes_raw),
    "behavior_synthesis": behavior_pass,
    "reflexivity": reflexivity_pass,
    "est_mapping": est_pass,
    "gmm": {k: v for k, v in gmm_out.items() if k != "cluster_labels"},
    "monte_carlo": mc_out,
    "triangulation_sources": [os.path.basename(s["path"]) for s in extra_sources],
}
with open(os.path.join(OUTPUT_DIR, "self_report_full_archive.json"), "w", encoding="utf-8") as f:
    json.dump(archive, f, ensure_ascii=False, indent=2)

print("[OK] LaTeX report, Markdown mirror, and JSON archive written.")
print("[OK] Files saved under:", OUTPUT_DIR)

In [ ]:
# -------------------------------------------------------------------------
# CELL 20 - Final summary & next steps
# -------------------------------------------------------------------------
print("=" * 66)
print("GROUND-TRUTH SELF-REPORT PIPELINE COMPLETE")
print("=" * 66)
files = sorted(os.listdir(OUTPUT_DIR))
for fname in files:
    full = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(full):
        print(f"  {fname:44s} {os.path.getsize(full) / 1024:10.1f} KB")

print("\nNEXT STEPS")
print("1. Copy thesis_self_report_ground_truth.tex into the thesis appendix.")
print("   It contains thematic + acoustic + EST + GMM + Monte-Carlo evidence.")
print("2. segment_embeddings.npy + claim_book_embeddings.npy feed the reserved G0/G1")
print("   Monte-Carlo replication tables and the MORE/Echo-Meter design simulation.")
print("3. Keep this ground-truth strand triangulated with the Uber corpus + survey + FGD.")
print("4. Honest boundary: this is reflexive/interpretive ground truth - readiness for")
print("   GMM/Monte-Carlo, NOT a human treatment effect or a population estimate.")

# How to read the outputs (paste-ready orientation)

## Main deliverables (in `thesis_ground_truth_output/` on Drive)

| File | Meaning |
| --- | --- |
| `thesis_self_report_ground_truth.tex` | Paste-ready appendix: thematic analysis (Braun & Clarke), acoustic/reflexive profile, EST-P1..P4 mapping, LOKLOK design rules, GMM + Monte-Carlo section. |
| `thesis_self_report_ground_truth.md` | Plain-text mirror of the same report for quick review. |
| `self_report_full_archive.json` | Single machine-readable archive of every stage (transcript, acoustics, themes, passes, GMM, MC, triangulation). |
| `segment_embeddings.npy` + `segment_meta.json` | 768-d per-utterance embedding matrix and metadata (feeds GMM and the G0/G1 Monte-Carlo replication). |
| `full_transcript_embedding.npy` | L2-normalised document-level embedding. |
| `claim_book_embeddings.npy` + `claim_book_meta.json` | Anchor lexicon (24 thesis ground-truth claims) used by the Echo-Meter / MORE Monte-Carlo. |
| `gmm_readiness.json` + `gmm_clusters_pca.png` | BIC-selected Gaussian-Mixture clusters and the PCA 2-D visualisation. |
| `monte_carlo_readiness.json` | Bootstrap cluster-share CIs + MC-SE + Echo-Meter trigger probabilities. |
| `researcher_self_report_transcript.json` | The timestamped, language-tagged transcript artefact. |
| `thematic_codes_raw.txt`, `themes_defined.json` | The open-coding log and final theme codebook (audit trail). |

## Workflow order in this notebook

1. Mount Drive → 2. transcribe → 3. acoustics → 4. embeddings → 5. optional triangulation sources →
6. load 7B LLM (auto-fallback to 3B) → 7. thematic open-coding → 8. theme definition →
9. behavioural synthesis → 10. reflexivity → 11. EST/design mapping → 12. GMM → 13. Monte Carlo →
14. report compilation.

## Raising tokens / detail

Edit `MAX_CHAPTER_TOKENS` in the configuration cell (default 1500) if a chapter run is cut short.
On a T4, keep values in the 1200–2500 range to balance depth and runtime.